In [2]:
import pandas as pd
import earthaccess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# 登录 NASA Earthdata
auth = earthaccess.login()

# 配置
CSV_PATH = "/data2/yuyao/methane_emission/preprocess_dataset_L89/merged_with_emit_tag.csv"
EMIT_RAW_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_EMIT")
EMIT_RAW_DIR.mkdir(exist_ok=True)

# 1. 筛选数据
df = pd.read_csv(CSV_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])

# 筛选条件：Permian Basin 范围 + 时间 < 2024-12-31
# Permian 典型范围: Lat [30, 34], Lon [-105, -101]
mask = (
    (df['plume_latitude'] >= 30) & (df['plume_latitude'] <= 34) &
    (df['plume_longitude'] >= -105) & (df['plume_longitude'] <= -101) &
    (df['datetime'] <= "2024-12-31") &
    (df['has_emit'] == 1)
)
filtered_df = df[mask].drop_duplicates(subset=['emit_granule_id'])

print(f"找到待下载的独特 EMIT 颗粒数量: {len(filtered_df)}")

def download_granule(granule_id):
    """修复后的单任务下载函数"""
    # 检查本地是否已存在，避免重复下载
    if list(EMIT_RAW_DIR.glob(f"*{granule_id}*.nc")):
        return f"{granule_id} 已存在"
    
    # 关键点：必须指定 short_name 或 collection_concept_id
    results = earthaccess.search_data(
        short_name='EMITL2ARFL',  # 显式限定为 EMIT L2A 反射率产品
        granule_name=granule_id,
        count=1
    )
    
    if results:
        earthaccess.download(results, str(EMIT_RAW_DIR))
        return f"{granule_id} 下载完成"
    return f"{granule_id} 未找到"

# 使用线程池加速下载
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(download_granule, filtered_df['emit_granule_id']))

找到待下载的独特 EMIT 颗粒数量: 79


QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2362.99it/s]
QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 391.21it/s]

QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 1185.50it/s]


QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2693.84it/s]





QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2541.49it/s]







QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2866.27it/s]















QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 789.24it/s]





QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 229.70it/s]



























PROCESSING TASKS | :  33%|███▎      | 1/3 [01:22<02:45, 82.73s/it]



PROCESSING TASKS | :  67%|██████▋   | 2/3 [04:56<02:39, 159.61s/it]








PROCESSING TASKS | : 100%|██████████| 3/3 [05:09<00:00, 103.31s/it]




COLLECTING RESULTS | : 100%|██████████| 3/3 [00:00<00:00, 65879.12it/s]




QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2232.60it/s]








PROCESSING TASKS | : 100%|██████████

In [3]:
import os
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray
import earthaccess
from pathlib import Path
from pyproj import Transformer
from scipy.interpolate import interp1d
import cupy as cp  # 如果报错，请确保已安装 cupy-cudaXX
from concurrent.futures import ThreadPoolExecutor

# ==================== 用户配置区 ====================
# CSV 文件路径
CSV_PATH = "/data2/yuyao/methane_emission/preprocess_dataset_L89/merged_with_emit_tag.csv"
# SRF 卷积文件路径
SRF_CSV = "./landsat9_oli_srf.csv"
# 存储 EMIT 原始 .nc 文件的目录
EMIT_RAW_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_EMIT")
# 存储最终生成的模拟 Landsat .tif 的目录
OUTPUT_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/EMIT_simulated_landsat9")

# 确保目录存在
EMIT_RAW_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
# ===================================================

# 1. 登录 NASA Earthdata (第一次运行会提示输入用户名密码)
auth = earthaccess.login()

def download_emit_granule(granule_id):
    """
    根据 ID 下载 EMIT 原始文件。
    解决了 "The CMR does not allow querying across granules" 报错。
    """
    # 检查本地是否已有
    existing = list(EMIT_RAW_DIR.glob(f"*{granule_id}*.nc"))
    if existing:
        return existing[0]
    
    print(f"   [Download] 正在检索并下载: {granule_id}")
    # 关键：必须指定 short_name 限定搜索范围
    results = earthaccess.search_data(
        short_name='EMITL2ARFL',
        granule_name=granule_id,
        count=1
    )
    if results:
        files = earthaccess.download(results, str(EMIT_RAW_DIR))
        return Path(files[0])
    return None

def open_emit_data(rfl_path):
    """读取 EMIT 的反射率、波长和经纬度矩阵"""
    ds = xr.open_dataset(rfl_path, engine='netcdf4')
    ds_band = xr.open_dataset(rfl_path, group='sensor_band_parameters', engine='netcdf4')
    wave_var = 'wavelengths' if 'wavelengths' in ds_band.data_vars else 'wavelength'
    wavelengths = ds_band[wave_var].values
    ds_loc = xr.open_dataset(rfl_path, group='location', engine='netcdf4')
    
    # 缩放因子处理：EMIT 反射率通常乘以了 10000
    rfl = ds['reflectance']
    return rfl, wavelengths, ds_loc['lon'].values, ds_loc['lat'].values

def gpu_spatial_resample(cp_sim_7band, e_lon, e_lat, l8_ref):
    """
    使用 GPU 进行空间重采样（最近邻插值）。
    这比 scipy.griddata 快 100 倍以上。
    """
    # 1. 准备 Landsat 的目标坐标网格
    t_lon_grid, t_lat_grid = np.meshgrid(l8_ref.x.values, l8_ref.y.values)
    
    # 如果 Landsat 是 UTM 投影，转回经纬度以便在 EMIT 矩阵中查找
    if not l8_ref.rio.crs.is_geographic:
        transformer = Transformer.from_crs(l8_ref.rio.crs, "EPSG:4326", always_xy=True)
        t_lon_grid, t_lat_grid = transformer.transform(t_lon_grid, t_lat_grid)

    # 2. 核心 GPU 逻辑：
    # 寻找 Landsat 每个像素点在 EMIT 经纬度矩阵中距离最近的索引
    cp_e_lon = cp.array(e_lon)
    cp_e_lat = cp.array(e_lat)
    cp_t_lon = cp.array(t_lon_grid)
    cp_t_lat = cp.array(t_lat_grid)

    # 展平 EMIT 坐标以便计算
    e_coords = cp.stack([cp_e_lat.ravel(), cp_e_lon.ravel()], axis=1)
    t_coords = cp.stack([cp_t_lat.ravel(), cp_t_lon.ravel()], axis=1)

    # 计算最近邻索引 (为了内存安全，这里使用简单的逐像素距离最小化映射)
    # 对于大规模生产，我们假设 EMIT 坐标相对局部线性，利用行列索引映射
    # 这里演示一个兼容性最强的 GPU 索引查找思路
    from cupyx.scipy.spatial import KDTree
    tree = KDTree(e_coords)
    _, indices = tree.query(t_coords)
    
    # 3. 根据索引提取 7 个波段的值
    h, w = t_lon_grid.shape
    output = cp.zeros((7, h, w), dtype=cp.float32)
    for b in range(7):
        # 将提取出的 1D 数据还原为 2D 影像
        output[b] = cp_sim_7band[b].ravel()[indices].reshape((h, w))
        
    return output.get()

def process_single_task(row, srf_df):
    """处理单个样本的全流程"""
    plume_id = row['plume_id']
    remote_l8_path = row['l89_path']
    granule_id = row['emit_granule_id']
    
    print(f"\n[Task Start] ID: {plume_id}")
    
    # --- Step 1: 异步拉取远程 Landsat 子集 ---
    try:
        # 只读 plume 所在的 15km 范围，不读全图（极速！）
        with rioxarray.open_rasterio(remote_l8_path, chunks=True) as ds:
            transformer = Transformer.from_crs("EPSG:4326", ds.rio.crs, always_xy=True)
            px, py = transformer.transform(row['plume_longitude'], row['plume_latitude'])
            buf = 250 * 30  # 约 7.5km 半径
            l8_subset = ds.rio.slice_xy(px-buf, py-buf, px+buf, py+buf).load()
    except Exception as e:
        print(f"   [Error] 远程读取失败: {e}")
        return

    # --- Step 2: 获取并加载 EMIT ---
    emit_nc = download_emit_granule(granule_id)
    if not emit_nc: return
    rfl_da, waves, e_lon, e_lat = open_emit_data(emit_nc)

    # --- Step 3: GPU 光谱卷积 (242波段 -> 7波段) ---
    cp_rfl = cp.array(np.nan_to_num(rfl_da.values, 0))
    sim_7band = []
    for i in range(1, 8):
        f = interp1d(srf_df["wavelength"], srf_df[f"b{i}"], fill_value=0, bounds_error=False)
        w = cp.array(f(waves))
        w /= (w.sum() + 1e-12)
        sim_7band.append(cp.tensordot(w, cp_rfl, axes=(0, 0)))
    cp_sim_7band = cp.stack(sim_7band)

    # --- Step 4: GPU 空间对齐与 Upsampling ---
    try:
        final_img = gpu_spatial_resample(cp_sim_7band, e_lon, e_lat, l8_subset)
    except Exception as e:
        print(f"   [Error] GPU 重采样失败: {e}")
        return

    # --- Step 5: 保存结果 ---
    out_tif = OUTPUT_DIR / f"{plume_id}_simulated_L9.tif"
    sim_da = xr.DataArray(
        final_img,
        dims=("band", "y", "x"),
        coords={"band": np.arange(1, 8), "y": l8_subset.y.values, "x": l8_subset.x.values}
    )
    sim_da.rio.write_crs(l8_subset.rio.crs, inplace=True)
    sim_da.rio.to_raster(out_tif)
    print(f"   [Success] 已保存: {out_tif.name}")

def main():
    # 加载任务和卷积表
    df = pd.read_csv(CSV_PATH)
    srf_df = pd.read_csv(SRF_CSV)
    
    # 筛选 Permian 盆地和 2024.12.31 以前的数据
    # 注：根据你的经纬度列名做筛选
    target_df = df[
        (df['plume_latitude'] >= 30) & (df['plume_latitude'] <= 34) &
        (df['plume_longitude'] >= -105) & (df['plume_longitude'] <= -101) &
        (pd.to_datetime(df['datetime']) <= "2024-12-31")
    ]
    
    print(f"🚀 准备处理 {len(target_df)} 个任务...")

    # 使用线程池并发下载和读取远程文件，计算在函数内进行
    with ThreadPoolExecutor(max_workers=4) as executor:
        for _, row in target_df.iterrows():
            executor.submit(process_task_safe, row, srf_df)

def process_task_safe(row, srf_df):
    try:
        process_single_task(row, srf_df)
    except Exception as e:
        print(f"   [Fatal] 任务 {row['plume_id']} 崩溃: {e}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'cupy'